# 5장 LLM API (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 5장과 관통 프로젝트 1단계에서 씁니다
import csv
import random
import re

with open(f'{DATA}/문의.csv', encoding='utf-8') as file:
    inquiries = list(csv.DictReader(file))        # 문의 60건. 열: 번호, 문의, 라벨, 긴급, 비고
LABELS = ['설비 이상', '품질', '안전', '문서와 데이터 요청']

# 100만 토큰당 단가(달러). Anthropic API의 기본 모델, 프롬프트 10만 토큰 이하 기준(2026-11 확인)
# 사내 API를 쓰면 사내 안내의 단가로 바꿉니다
PRICES = {'소형': {'input': 0.10, 'output': 0.50, 'cache_write': 0.125, 'cache_read': 0.01},
          '기본': {'input': 2.00, 'output': 10.00, 'cache_write': 2.50, 'cache_read': 0.10},
          '대형': {'input': 4.00, 'output': 20.00, 'cache_write': 5.00, 'cache_read': 0.20}}


def cost(reply, tier):
    """응답 하나의 요금(달러). 입력 토큰과 출력 토큰(thinking 포함)에 각각의 단가를 곱해 더합니다"""
    price = PRICES[tier]
    return (reply.input_tokens * price['input'] + reply.output_tokens * price['output']) / 1_000_000


def numbered(items):
    """문의 목록을 "번호: 문의" 줄로 이어 붙입니다"""
    return '\n'.join(f"{item['번호']}: {item['문의']}" for item in items)


def parse_labels(text):
    """응답에서 "Q01: 라벨" 형식의 줄을 찾아 {번호: 라벨}로 돌려줍니다"""
    found = {}
    for line in text.splitlines():
        match = re.match(r'\s*[-*]?\s*(Q\d{2})\s*[:：]\s*(.+?)\s*$', line)
        if match:
            found[match.group(1)] = match.group(2).strip()
    return found


def accuracy(predicted, items):
    """정답 라벨과 같은 건의 비율"""
    return sum(predicted.get(item['번호']) == item['라벨'] for item in items) / len(items)


print(f'문의 {len(inquiries)}건, 라벨 {LABELS}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

문의 60건, 라벨 ['설비 이상', '품질', '안전', '문서와 데이터 요청']
소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 5-1. SDK로 직접 보내기

교재의 절: "thinking의 양을 정하는 effort"

같은 계산 문제를 SDK로 직접 보내고, 응답의 블록 종류, 종료 이유, 입력 토큰, 출력 토큰과 그중 thinking 토큰을 봅니다. Anthropic 쪽은 effort를 `low`와 `high`로 바꿔 두 번 보내고 출력 토큰이 어떻게 다른지 봅니다. 두 코드 가운데 지금 쓰는 API에 해당하는 쪽만 실행되고, 이 셀은 API를 2번(OpenAI 호환은 1번) 호출합니다.

**Anthropic**

**OpenAI 호환**

In [3]:
question = 'UPH 1,200인 와이어 본더 3대가 하루 22시간, 가동률 87%로 돌면 하루 생산량은 몇 개입니까? 숫자만 답하세요.'
if llm.provider() == 'anthropic':
    import anthropic
    client = anthropic.Anthropic()                    # 환경 변수 ANTHROPIC_API_KEY를 읽습니다
    for effort in ['low', 'high']:
        response = client.messages.create(
            model=llm.model_name('소형'), max_tokens=4000,   # max_tokens에는 thinking 토큰도 들어갑니다
            system='너는 후공정 라인의 생산 계산을 돕는 도우미다.',
            messages=[{'role': 'user', 'content': question}],
            output_config={'effort': effort})        # Anthropic API에만 있는 설정입니다
        usage, text = response.usage, ''.join(block.text for block in response.content if block.type == 'text')
        thinking = usage.output_tokens_details.thinking_tokens if usage.output_tokens_details else '알 수 없음'
        print(f'effort={effort}: 블록 {[block.type for block in response.content]}, 종료 이유 {response.stop_reason}')
        print(f'  입력 {usage.input_tokens}토큰, 출력 {usage.output_tokens}토큰(그중 thinking {thinking}) → {text.strip()}')
else:
    print('Anthropic을 쓰지 않으므로 건너뜁니다.')

effort=low: 블록 ['thinking', 'text'], 종료 이유 end_turn
  입력 103토큰, 출력 70토큰(그중 thinking 66) → 68904


effort=high: 블록 ['thinking', 'text'], 종료 이유 end_turn
  입력 103토큰, 출력 163토큰(그중 thinking 159) → 68904


In [4]:
question = 'UPH 1,200인 와이어 본더 3대가 하루 22시간, 가동률 87%로 돌면 하루 생산량은 몇 개입니까? 숫자만 답하세요.'
if llm.provider() == 'openai':
    from openai import OpenAI
    client = OpenAI(base_url=os.environ['OPENAI_BASE_URL'], api_key=os.environ['OPENAI_API_KEY'])
    response = client.chat.completions.create(
        model=llm.model_name('소형'), max_tokens=4000,
        messages=[{'role': 'system', 'content': '너는 후공정 라인의 생산 계산을 돕는 도우미다.'},
                  {'role': 'user', 'content': question}])      # 시스템 프롬프트는 맨 앞의 system 메시지입니다
    choice, usage = response.choices[0], response.usage
    details = usage.completion_tokens_details
    thinking = details.reasoning_tokens if details and details.reasoning_tokens is not None else '알 수 없음'
    print(f'종료 이유 {choice.finish_reason}, 입력 {usage.prompt_tokens}토큰, '
          f'출력 {usage.completion_tokens}토큰(그중 thinking {thinking}) → {choice.message.content.strip()}')
else:
    print('OpenAI 호환 API를 쓰지 않으므로 건너뜁니다.')

OpenAI 호환 API를 쓰지 않으므로 건너뜁니다.


## 예제 5-2. 등급별 정확도, 시간, 요금

교재의 절: "모델 등급 고르기"

4장의 문의 데이터에서 두 라벨에 걸쳐 헷갈리는 문의 15건(Q41~Q55)을 한 요청에 담아 소형, 기본, 대형 모델에 보냅니다. 보내기 전에 센 입력 토큰과 실제 입력 토큰, 정확도, 출력 토큰, 걸린 시간, 요금을 비교합니다. 이 셀은 API를 3번 호출하고, 토큰 세기를 3번 부릅니다(요금 없음).

In [5]:
hard = inquiries[40:55]                                # Q41~Q55, 두 라벨에 걸쳐 헷갈리는 문의
system = ('문의를 다음 판정 기준을 위에서부터 차례로 적용해 하나의 라벨로 분류한다. '
          '1. 사람이 다치거나 다칠 위험이 있으면 안전 2. 원하는 것이 문서나 기록의 전달이면 문서와 데이터 요청 '
          '3. 원인 설비가 특정되고 조치 대상이 설비이면 설비 이상 4. 제품 결과에 이상이 있고 원인 설비가 특정되지 않으면 품질. '
          '답은 "번호: 라벨" 형식으로 한 줄에 한 건씩만 쓴다.')
messages = [{'role': 'user', 'content': numbered(hard)}]
for tier in ['소형', '기본', '대형']:
    counted = llm.count_tokens(messages, system=system, model=tier)       # 보내기 전에 셉니다. OpenAI 호환이면 None
    reply = llm.send(messages, system=system, model=tier, max_tokens=16000, timeout=300)
    print(f'{tier}: 정확도 {accuracy(parse_labels(reply.text), hard):.0%}, 입력 {counted}→{reply.input_tokens}토큰, '
          f'출력 {reply.output_tokens:,}토큰, {reply.seconds:.1f}초, {cost(reply, tier):.4f}달러')

소형: 정확도 93%, 입력 912→912토큰, 출력 1,475토큰, 6.7초, 0.0008달러


기본: 정확도 93%, 입력 912→912토큰, 출력 1,348토큰, 10.3초, 0.0153달러


대형: 정확도 93%, 입력 912→912토큰, 출력 870토큰, 8.5초, 0.0210달러


## 예제 5-3. 세 가지 실패

교재의 절: "실패 유형"

없는 모델 이름, 0.5초 타임아웃, 20토큰의 최대 출력 토큰으로 같은 요청을 보냅니다. 앞의 두 요청이 어떤 예외로 바뀌는지, 셋째 요청은 예외 없이 무엇을 돌려주는지 봅니다. 이 셀은 API를 최대 3번 호출합니다. 재시도가 섞이지 않도록 `llm.send()`로 한 번씩만 보냅니다.

In [6]:
messages = [{'role': 'user', 'content': '와이어 본딩 공정을 세 문장으로 설명해 주세요.'}]
cases = {'없는 모델 이름': dict(model='claude-no-such-model'),
         '타임아웃 0.5초': dict(model='소형', timeout=0.5),        # 0.5초 안에 응답이 오지 않으면 포기합니다
         '최대 출력 토큰 20': dict(model='소형', max_tokens=20)}    # thinking 토큰도 이 20토큰 안에 들어갑니다
for name, options in cases.items():
    try:
        reply = llm.send(messages, **options)
        print(f'{name}: 예외 없음, 종료 이유 {reply.stop_reason}, 출력 {reply.output_tokens}토큰, 응답 텍스트 {reply.text!r}')
    except llm.CallError as error:
        print(f'{name}: 예외 {error.kind}, 상태 코드 {error.status}, 재시도할 오류인지 {error.retryable}')

없는 모델 이름: 예외 not_found, 상태 코드 404, 재시도할 오류인지 False


타임아웃 0.5초: 예외 timeout, 상태 코드 None, 재시도할 오류인지 True


최대 출력 토큰 20: 예외 없음, 종료 이유 max_tokens, 출력 20토큰, 응답 텍스트 '와이어 본딩은 반도체 칩의 본딩'


## 실습 5-1. 한 달 요금 어림하기

교재의 절: "개념 확인 실습"

문의를 한 건에 한 요청씩 한 달에 1만 건 분류합니다. 요청마다 시스템 프롬프트 800토큰과 문의 60토큰을 보내고, 출력은 thinking을 포함해 평균 40토큰이라고 가정합니다. 등급마다 한 달 요금을 계산하는 `monthly_cost()`를 작성하고, 시스템 프롬프트를 캐싱할 때의 요금도 계산합니다. 캐싱은 첫 요청에서 캐시에 쓰고 나머지 요청은 모두 캐시에서 읽는다고 가정합니다. 먼저 `my_guess`에 대형 모델의 두 요금을 어림해 적습니다. API를 호출하지 않습니다.

In [7]:
def monthly_cost(tier, requests=10_000, system_tokens=800, inquiry_tokens=60, output_tokens=40, cached=False):
    """한 달 요금(달러). PRICES의 단가를 씁니다. cached=True면 시스템 프롬프트를 캐싱합니다"""
    price = PRICES[tier]
    if cached:
        system_cost = system_tokens * (price['cache_write'] + (requests - 1) * price['cache_read'])
    else:
        system_cost = requests * system_tokens * price['input']
    rest = requests * (inquiry_tokens * price['input'] + output_tokens * price['output'])
    return (system_cost + rest) / 1_000_000


my_guess = {'대형, 캐싱 없음': 50, '대형, 캐싱': 20}
for tier in PRICES:
    print(f'{tier}: 캐싱 없음 {monthly_cost(tier):,.2f}달러, 캐싱 {monthly_cost(tier, cached=True):,.2f}달러')
print('예측:', my_guess)

소형: 캐싱 없음 1.06달러, 캐싱 0.34달러
기본: 캐싱 없음 21.20달러, 캐싱 6.00달러
대형: 캐싱 없음 42.40달러, 캐싱 12.00달러
예측: {'대형, 캐싱 없음': 50, '대형, 캐싱': 20}
